In [1]:
import pandas as pd
import hashlib
import json
from pathlib import Path
from datetime import datetime

In [2]:
# import shutil
# from pathlib import Path
# from datetime import datetime

# # Paths
# HISTORY_PATH = Path("selection_history.json")
# TRAIN_DIR    = Path("training_sets")
# OUTPUT_DIR   = Path(".")

# # Backup folder with timestamp
# BACKUP_DIR = Path("backup_before_reset") / datetime.now().strftime("%Y%m%d_%H%M%S")
# BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# # Move history file if present
# if HISTORY_PATH.exists():
#     shutil.move(str(HISTORY_PATH), BACKUP_DIR / HISTORY_PATH.name)

# # Move previous training sets
# if TRAIN_DIR.exists():
#     shutil.move(str(TRAIN_DIR), BACKUP_DIR / TRAIN_DIR.name)

# # Move any unique_sample_*.csv files
# moved_any = False
# for p in OUTPUT_DIR.glob("unique_sample_*.csv"):
#     shutil.move(str(p), BACKUP_DIR / p.name)
#     moved_any = True

# print("✅ Reset complete. Archived prior state to:", BACKUP_DIR.resolve())

import shutil
from pathlib import Path
from datetime import datetime

# Paths (fixed leading slashes)
HISTORY_PATH = Path("/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/selection_history.json")
TRAIN_DIR    = Path("/home/ubuntu/TW_MultiLabel_SMP/datasets/training_sets")
OUTPUT_DIR   = Path(".")

# Backup folder with timestamp
BACKUP_DIR = Path("/home/ubuntu/TW_MultiLabel_SMP/datasets/backup_before_reset") / datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# ---- Preserve history (copy, don't move) ----
if HISTORY_PATH.exists():
    backup_history = BACKUP_DIR / HISTORY_PATH.name
    try:
        shutil.copy2(HISTORY_PATH, backup_history)
        print(f"📝 Preserved history: copied to {backup_history}")
    except Exception as e:
        print(f"⚠️ Could not copy history file: {e}")
else:
    print("ℹ️ No selection_history.json found to preserve.")

# ---- Move previous training sets to backup ----
if TRAIN_DIR.exists():
    dest = BACKUP_DIR / TRAIN_DIR.name
    try:
        shutil.move(str(TRAIN_DIR), dest)
        print(f"📦 Archived training_sets to: {dest}")
    except Exception as e:
        print(f"⚠️ Could not move training_sets: {e}")
else:
    print("ℹ️ No training_sets directory found to archive.")

# ---- Move any unique_sample_*.csv files to backup ----
moved_any = False
for p in OUTPUT_DIR.glob("unique_sample_*.csv"):
    try:
        shutil.move(str(p), BACKUP_DIR / p.name)
        print(f"📄 Archived {p.name}")
        moved_any = True
    except Exception as e:
        print(f"⚠️ Could not move {p}: {e}")

if not moved_any:
    print("ℹ️ No unique_sample_*.csv files found to archive.")

print("✅ Reset complete. Old artifacts archived. History preserved in place.")



📝 Preserved history: copied to /home/ubuntu/TW_MultiLabel_SMP/datasets/backup_before_reset/20251018_212211/selection_history.json
ℹ️ No training_sets directory found to archive.
📄 Archived unique_sample_20251016_174914.csv
✅ Reset complete. Old artifacts archived. History preserved in place.


In [3]:
def row_hash(row: pd.Series) -> str:
    """Generate a deterministic hash of a row if no explicit ID column exists."""
    obj = row.to_dict()
    normalized = {str(k): ("" if pd.isna(v) else str(v)) for k, v in obj.items()}
    payload = json.dumps(normalized, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def load_df(path: str, dataset_name: str, id_column: str = None) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df["__dataset"] = dataset_name
    df["__source_file"] = Path(path).name

    if id_column and id_column in df.columns:
        df["unique_key"] = df[id_column].astype(str)
    else:
        df["unique_key"] = df.apply(row_hash, axis=1)

    return df


In [7]:
# Update these paths to your local copies
ABORTION_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/abortion_data-updated - new_abortion_related_subreddits_text_posts .csv"
MISCARRIAGE_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/miscarriage-data-updated - miscarriage_related_posts.csv"
HARASSMENT_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/sexual-harrassment-data-updated - RelevantByTitle.csv"

# History file (persists across runs)
HISTORY_PATH = Path("/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/selection_history.json")

# Desired counts per dataset
counts = {
    "abortion": 173,
    "miscarriage": 154,
    "harassment": 173
}

# Optional: if your CSVs have a post_id or id column
ID_COLUMN = None   # e.g. "post_id"


In [8]:
# Load CSVs
abortion_df = load_df(ABORTION_PATH, "abortion", ID_COLUMN)
miscarriage_df = load_df(MISCARRIAGE_PATH, "miscarriage", ID_COLUMN)
harassment_df = load_df(HARASSMENT_PATH, "harassment", ID_COLUMN)

# Load or initialize selection history
if HISTORY_PATH.exists():
    with open(HISTORY_PATH, "r", encoding="utf-8") as f:
        history = json.load(f)
else:
    history = {"used_keys": [], "runs": []}

used_keys = set(history.get("used_keys", []))

# Exclude previously used posts
def exclude_used(df):
    return df[~df["unique_key"].isin(used_keys)].copy()

ab_pool = exclude_used(abortion_df)
mi_pool = exclude_used(miscarriage_df)
sh_pool = exclude_used(harassment_df)

In [9]:
shortages = []
if len(ab_pool) < counts["abortion"]:
    shortages.append(f"abortion (need {counts['abortion']}, have {len(ab_pool)})")
if len(mi_pool) < counts["miscarriage"]:
    shortages.append(f"miscarriage (need {counts['miscarriage']}, have {len(mi_pool)})")
if len(sh_pool) < counts["harassment"]:
    shortages.append(f"harassment (need {counts['harassment']}, have {len(sh_pool)})")

if shortages:
    raise RuntimeError("Not enough fresh rows: " + "; ".join(shortages))

sample_ab = ab_pool.sample(n=counts["abortion"], replace=False, random_state=None)
sample_mi = mi_pool.sample(n=counts["miscarriage"], replace=False, random_state=None)
sample_sh = sh_pool.sample(n=counts["harassment"], replace=False, random_state=None)

sample_all = pd.concat([sample_ab, sample_mi, sample_sh], ignore_index=True)
sample_all = sample_all.sample(frac=1.0).reset_index(drop=True)  # shuffle


In [10]:
# Save timestamped CSV
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = Path(f"unique_sample_{ts}.csv")
sample_all.to_csv(out_path, index=False)

# Update history
new_keys = sample_all["unique_key"].tolist()
history["used_keys"].extend(new_keys)
history["runs"].append({
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "output_file": str(out_path),
    "counts": counts,
    "selected": len(new_keys)
})

with open(HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {len(sample_all)} posts to {out_path}")
print(f"Remaining after this run:")
print("  abortion:", len(ab_pool) - counts["abortion"])
print("  miscarriage:", len(mi_pool) - counts["miscarriage"])
print("  harassment:", len(sh_pool) - counts["harassment"])


✅ Saved 500 posts to unique_sample_20251018_212534.csv
Remaining after this run:
  abortion: 3403
  miscarriage: 0
  harassment: 4154


In [11]:
# Make a clean training label column (good for ML pipelines)
sample_all = sample_all.copy()
sample_all["label"] = sample_all["__dataset"]  # keep your original columns intact

# Create a training_sets folder
TRAIN_DIR = Path("training_sets")
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

# Save a per-run training file (500 rows)
train_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
train_csv = TRAIN_DIR / f"train_{train_ts}.csv"
sample_all.to_csv(train_csv, index=False)

# Also keep a stable "latest" pointer you can reference in code
latest_csv = TRAIN_DIR / "train_latest.csv"
sample_all.to_csv(latest_csv, index=False)

print(f"✅ Saved training set (500 rows): {train_csv}")
print(f"🔁 Also updated: {latest_csv}")

# (Optional) Save per-class training files for class-specific experiments
PER_CLASS_DIR = TRAIN_DIR / f"per_class_{train_ts}"
PER_CLASS_DIR.mkdir(parents=True, exist_ok=True)

for cls in sample_all["label"].unique():
    out_cls = PER_CLASS_DIR / f"{cls}_train_{train_ts}.csv"
    sample_all[sample_all["label"] == cls].to_csv(out_cls, index=False)
    print(f"• Saved {cls} subset to: {out_cls}")

# (Optional) Keep a cumulative union of everything ever sampled (good for audit/repro)
CUMULATIVE_CSV = TRAIN_DIR / "all_selected_so_far.csv"
if CUMULATIVE_CSV.exists():
    prev = pd.read_csv(CUMULATIVE_CSV, low_memory=False)
    # Use unique_key to de-dup
    combined = pd.concat([prev, sample_all], ignore_index=True)
    combined = combined.drop_duplicates(subset=["unique_key"])
else:
    combined = sample_all

combined.to_csv(CUMULATIVE_CSV, index=False)
print(f"📚 Cumulative selected-so-far updated: {CUMULATIVE_CSV}")


✅ Saved training set (500 rows): training_sets/train_20251018_212601.csv
🔁 Also updated: training_sets/train_latest.csv
• Saved abortion subset to: training_sets/per_class_20251018_212601/abortion_train_20251018_212601.csv
• Saved miscarriage subset to: training_sets/per_class_20251018_212601/miscarriage_train_20251018_212601.csv
• Saved harassment subset to: training_sets/per_class_20251018_212601/harassment_train_20251018_212601.csv
📚 Cumulative selected-so-far updated: training_sets/all_selected_so_far.csv


In [12]:
sample_all.head(10)

,id,subreddit,title,selftext,created_utc,url,Tags,__dataset,__source_file,unique_key,label
0,1kfpv1l,OBGYN,3 Questions about progesterone and menopause,I’m in menopause. Had some post-meno bleeding ...,2025-05-05 23:11:44,https://www.reddit.com/r/obgyn/comments/1kfpv1...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,fa0dbe1ac2ffcc1724ca0895c06ae0ce6796ad9bea5a8f...,abortion
1,1ljp5wz,BabyBumps,Quit eating my f$#%ing food!,On the verge of crashing out over this. My 30 ...,2025-06-24 22:34:25,https://www.reddit.com/r/BabyBumps/comments/1l...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,7860915380c4a2e1ef4a3820dbd22b7882a5f2a18d2f76...,abortion
2,1kxz0r8,Miscarriage,"1 MMC, 1 failed Misoprostol, Septic miscarriag...","At the end of April, my husband and I were at ...",2025-05-29 1:57:59,https://www.reddit.com/r/Miscarriage/comments/...,NaN,miscarriage,miscarriage-data-updated - miscarriage_related...,a5cbfa45f88d6eb36e8f0007cb49b83156bcd64df81351...,miscarriage
3,1danfpl,therapy,Therapist red flag? Or doing his job?,My (25F) partner (26M) sees a male CSAT (calli...,2024-06-07 22:17:33,https://www.reddit.com/r/therapy/comments/1dan...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,e7c463968340efdea7a8025abdbdb395dd35ad88ef33e4...,abortion
4,1lknllw,Miscarriage,Anyone lose a healthy baby around 16 weeks?!,\nI lost my baby girl at 16 weeks pregnant in ...,2025-06-26 1:34:22,https://www.reddit.com/r/Miscarriage/comments/...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,3ae7c91d49917c479c88f4673d048103bcc4ac2e789e49...,abortion
5,oeqa4k,assault,"I only just realized I was a victim, and now I...",Need to rant about my story\nTw \nIt happened ...,2021-07-06 7:44:34,https://www.reddit.com/r/sexualassault/comment...,NaN,harassment,sexual-harrassment-data-updated - RelevantByTi...,41924bd2eb7315c6671fb0bdec15b6dc12c45eb2984300...,harassment
6,1l4zs8v,abortion,Increased clotting and nausea,"I took mifepristone on Monday, my misoprostol ...",2025-06-06 18:30:31,https://www.reddit.com/r/abortion/comments/1l4...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,112ab47d8f1201171a721a249c2610ecf5a66f39d19bda...,abortion
7,okn0uc,assault,Finally told somebody my brother molested me b...,So I've always been a very open book you can s...,2021-07-15 6:26:28,https://www.reddit.com/r/sexualassault/comment...,NaN,harassment,sexual-harrassment-data-updated - RelevantByTi...,fbe8ff6bba114543d33a6e1fcea0fa98592b74d8d5a911...,harassment
8,7enzud,assault,I was sexually assaulted by a female in the Un...,I was a Master at Arms in the United States Na...,2017-11-22 3:58:40,https://www.reddit.com/r/sexualassault/comment...,NaN,harassment,sexual-harrassment-data-updated - RelevantByTi...,ce4f17413e4cf77ca8e93a21edbb6fed7448d15fa7ad84...,harassment
9,1gwus8j,Pregnant,"At 13 weeks pregnant without an ultrasound, en...",I’m feeling pissed off more than sad. \nMy hub...,2024-11-22 0:22:46,https://www.reddit.com/r/pregnant/comments/1gw...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,c3b93370d4fb8cb446db4a14963ed81fee062bde0a503f...,abortion
